# VKM Recommender System 3.0: EDA-Driven & Menselijk

## Inleiding
Dit is de derde iteratie van het aanbevelingssysteem voor VKM-modules. Deze versie bouwt voort op eerdere versies maar integreert cruciale inzichten uit de **Exploratory Data Analysis (EDA)**.

**Verbeteringen in versie 3.0 (gebaseerd op EDA inzichten):**
*   **Slimmere Locatie-filtering:** Uit de EDA bleek dat de meeste modules in Breda of Den Bosch zijn. In plaats van een harde filter, geven we nu een **bonus** aan modules in je voorkeursstad. Zo mis je geen perfecte match in een andere stad.
*   **Betere Moeilijkheidsgraad:** De gemiddelde moeilijkheid is 3.16. We straffen nu minder hard als een module iets moeilijker is dan je voorkeur, tenzij het verschil te groot is.
*   **Focus op Tags:** Om mismatches zoals 'Informatica vs Forensisch' te voorkomen, wegen tags (zoals 'programmeren', 'zorg') zwaarder mee.
*   **Menselijke Uitleg:** De redenen waarom een module wordt aanbevolen zijn nu in begrijpelijk Nederlands.

## 1. Setup & Libraries
We gebruiken standaard libraries om de code robuust en portable te houden.

In [1]:
import pandas as pd
import numpy as np
import re
import ast
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Robuuste stopwords afhandeling
try:
    import nltk
    from nltk.corpus import stopwords
    nltk.download('stopwords', quiet=True)
    dutch_stopwords = stopwords.words('dutch')
except (ImportError, LookupError):
    # Fallback als NLTK niet werkt
    dutch_stopwords = ['de', 'en', 'van', 'ik', 'te', 'dat', 'die', 'in', 'een', 'hij', 'het', 'niet', 'zijn', 'is', 'was', 'op', 'aan', 'met', 'als', 'voor', 'had', 'er', 'maar', 'om', 'hem', 'dan', 'zou', 'of', 'wat', 'mijn', 'men', 'dit', 'zo', 'door', 'over', 'ze', 'zich', 'bij', 'ook', 'tot', 'je', 'mij', 'uit', 'der', 'daar', 'haar', 'naar', 'heb', 'hoe', 'heeft', 'hebben', 'deze', 'u', 'want', 'nog', 'zal', 'me', 'zij', 'nu', 'ge', 'geen', 'omdat', 'iets', 'worden', 'toch', 'al', 'waren', 'veel', 'meer', 'doen', 'toen', 'moet', 'ben', 'zonder', 'kan', 'hun', 'dus', 'alles', 'onder', 'ja', 'eens', 'hier', 'wie', 'werd', 'altijd', 'doch', 'wordt', 'wezen', 'kunnen', 'ons', 'zelf', 'tegen', 'na', 'reeds', 'wil', 'kon', 'niets', 'uw', 'iemand', 'geweest', 'andere']

# Extra domein-specifieke stopwoorden uit EDA
dutch_stopwords.extend(['student', 'studenten', 'module', 'minor', 'leerdoelen', 'vak', 'cursus', 'project', 'opdracht', 'werken', 'leren'])

print("Setup voltooid.")

Setup voltooid.


## 2. Data Laden & Opschonen
We laden de dataset en zorgen dat de tags correct worden ingelezen (van string naar lijst).

In [2]:
DATA_PATH = "Opgeschoonde_VKM_dataset.csv"

def load_and_clean_data(path):
    df = pd.read_csv(path)
    
    # Kolomnamen normaliseren
    df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]
    
    # Tags parsen (staan vaak als string "['tag1', 'tag2']" in CSV)
    def parse_tags(tag_str):
        try:
            if pd.isna(tag_str):
                return []
            # Als het al een lijst is, prima
            if isinstance(tag_str, list):
                return tag_str
            # Probeer literal eval voor string representatie van lijst
            return ast.literal_eval(tag_str)
        except:
            # Fallback: simpele split als eval faalt
            return str(tag_str).replace("[", "").replace("]", "").replace("'", "").split(",")

    if "module_tags" in df.columns:
        df["parsed_tags"] = df["module_tags"].apply(parse_tags)
    else:
        df["parsed_tags"] = [[] for _ in range(len(df))]
        
    return df

df = load_and_clean_data(DATA_PATH)
print(f"Dataset geladen: {len(df)} modules.")
print("Voorbeeld tags:", df["parsed_tags"].iloc[0])

Dataset geladen: 211 modules.
Voorbeeld tags: ['brein', 'gedragsbeinvloeding', 'ontwikkelingspsychologie', 'gespreksvoering', 'en', 'ontwikkelingsfasen']


## 3. Feature Engineering: Slimme Tekstverwerking
Om te voorkomen dat 'Informatica' studenten 'Forensisch' krijgen aangeraden, geven we extra gewicht aan de **Naam** en de **Tags** van een module. De beschrijving is belangrijk, maar de titel en tags bevatten vaak de kern.

In [3]:
# We maken een 'weighted_text' kolom
# Naam telt 3x mee, Tags tellen 3x mee, Beschrijving 1x
def create_weighted_text(row):
    name = str(row.get("name", ""))
    desc = str(row.get("description", ""))
    tags = " ".join(row.get("parsed_tags", []))
    
    # Herhaal belangrijke velden
    weighted = (name + " ") * 3 + (tags + " ") * 3 + desc
    return weighted

df["weighted_text"] = df.apply(create_weighted_text, axis=1)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words=dutch_stopwords
)
tfidf_matrix = tfidf.fit_transform(df["weighted_text"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (211, 5000)


## 4. Het Profiel & De Matcher
Hier definiëren we het studentprofiel en de logica om de beste match te vinden. 

### EDA Inzichten toegepast:
1.  **Locatie:** Omdat de meeste modules in Breda/Den Bosch zijn, sluiten we andere locaties niet hard uit, maar geven we voorkeurslocaties een **bonus**.
2.  **Moeilijkheid:** Gemiddelde is 3.16. We zijn flexibel, maar waarschuwen als het niveau te hoog is.
3.  **Tags:** Exacte tag matches geven een grote score boost.

In [4]:
@dataclass
class CandidateProfile:
    interests_text: str
    preferred_location: Optional[str] = None
    min_studycredits: Optional[float] = None
    max_difficulty: Optional[float] = None
    moduletags_include: Optional[List[str]] = None

def calculate_match(row, profile: CandidateProfile) -> Tuple[float, List[str]]:
    """
    Berekent een score-aanpassing (bonus/straf) en genereert uitleg.
    """
    reasons = []
    
    # 1. Locatie Check (Soft Boost ipv Hard Filter - EDA Inzicht)
    # Uit EDA blijkt dat Breda/Den Bosch dominant zijn. Hard filteren is te streng.
    loc_bonus = 0.0
    loc_col = "location" if "location" in row.index else None
    if profile.preferred_location and loc_col:
        # Case-insensitive check
        if str(profile.preferred_location).lower() in str(row[loc_col]).lower():
            loc_bonus = 0.15  # Bonus voor perfecte locatie match
            reasons.append(f"✅ **Locatie**: Deze module is in {row[loc_col]}, wat je voorkeur heeft.")
        else:
            # Geen straf, maar ook geen bonus. Wel een notitie.
            reasons.append(f"📍 **Locatie**: Let op, deze module is in {row[loc_col]} (je voorkeur was {profile.preferred_location}).")

    # 2. Module Tags / Domain Match (Cruciaal voor 'Informatica vs Forensisch')
    tag_bonus = 0.0
    row_tags = row.get("parsed_tags", [])
    if profile.moduletags_include:
        # Zoek naar exacte matches in de tags
        matches = [tag for tag in profile.moduletags_include if any(tag.lower() in t.lower() for t in row_tags)]
        if matches:
            tag_bonus = 0.25 * (len(matches) / len(profile.moduletags_include)) # Flinke bonus voor tag matches
            reasons.append(f"🎯 **Onderwerp**: Matcht met je interesse in: {', '.join(matches)}.")
        else:
            # Als de gebruiker specifieke tags zocht maar ze niet vond, is dat een gemis
            pass 

    # 3. Difficulty (Refined based on EDA avg 3.16)
    diff_penalty = 0.0
    if profile.max_difficulty and "estimated_difficulty" in row:
        try:
            diff = float(row["estimated_difficulty"])
            if diff > profile.max_difficulty:
                # Straf is proportioneel aan het verschil, maar niet direct fataal
                diff_penalty = (diff - profile.max_difficulty) * 0.1
                reasons.append(f"⚠️ **Niveau**: Iets uitdagender ({diff}/5) dan je limiet ({profile.max_difficulty}/5).")
        except:
            pass

    # 4. Study Credits (Hard constraint blijft nuttig)
    credits_penalty = 0.0
    if profile.min_studycredits and "study_credit" in row:
        try:
            creds = float(row["study_credit"])
            if creds < profile.min_studycredits:
                credits_penalty = 0.2 # Flinke straf voor te weinig punten
                reasons.append(f"⚠️ **Punten**: Deze module is {creds} EC, je zocht {profile.min_studycredits} EC.")
        except:
            pass

    # Totale score aanpassing
    # Base score is content similarity (0-1). We tellen bonussen op en trekken straffen af.
    # We cappen het resultaat niet hard op 1.0, zodat super-matches eruit springen.
    final_modifier = loc_bonus + tag_bonus - diff_penalty - credits_penalty
    
    return final_modifier, reasons

def recommend(profile: CandidateProfile, k=5):
    # 1. Content Similarity (TF-IDF)
    # We voegen de tags van het profiel toe aan de interesse-tekst voor de vectorizer
    query_text = profile.interests_text
    if profile.moduletags_include:
        query_text += " " + " ".join(profile.moduletags_include)
        
    query_vec = tfidf.transform([query_text])
    content_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # 2. Popularity (Normalized)
    pop_col = "popularity_score" if "popularity_score" in df.columns else None
    if pop_col:
        pop_raw = df[pop_col].fillna(0)
        pop_score = (pop_raw - pop_raw.min()) / (pop_raw.max() - pop_raw.min() + 1e-9)
    else:
        pop_score = pd.Series(0, index=df.index)

    rows = []
    title_col = "name" if "name" in df.columns else df.columns[0]
    loc_col = "location" if "location" in df.columns else ""

    for idx, row in df.iterrows():
        c_score = float(content_scores[idx])
        
        # Calculate Final Weighted Score
        # Start with content similarity (0-1)
        base_score = c_score
        
        # Apply modifiers (bonuses and penalties)
        modifier, reasons = calculate_match(row, profile)
        
        # Popularity boost (small)
        pop_boost = pop_score[idx] * 0.05
        
        final_score = base_score + modifier + pop_boost
        
        # Generate explanation text
        explanation = " ".join(reasons)
        if not explanation:
            explanation = "Deze module past goed bij je interesses op basis van de beschrijving."

        rows.append({
            "index": idx,
            "name": row.get(title_col, ""),
            "location": row.get(loc_col, ""),
            "final_score": final_score,
            "content_sim": c_score,
            "explanation": explanation,
            "tags": row.get("parsed_tags", [])
        })

    # Create DataFrame and sort by Final Score (highest first)
    rec_df = pd.DataFrame(rows).sort_values("final_score", ascending=False).head(k).reset_index(drop=True)
    return rec_df

## 5. Demo & Resultaten
Laten we het systeem testen met een specifiek profiel.

In [5]:
# Test Profiel: Informatica student die in Breda wil studeren
student_profile = CandidateProfile(
    interests_text="Ik hou van programmeren, software development, data analysis en AI. Ik wil graag technische skills leren.",
    preferred_location="Breda",
    min_studycredits=15,
    max_difficulty=4,
    moduletags_include=["software", "data", "programmeren"] # Expliciete tags helpen enorm
)

recommendations = recommend(student_profile, k=5)

print("\n--- Aanbevolen Modules ---\n")
for i, row in recommendations.iterrows():
    print(f"{i+1}. {row['name']} (Score: {row['final_score']:.2f})")
    print(f"   Locatie: {row['location']}")
    print(f"   Tags: {row['tags']}")
    print(f"   Waarom: {row['explanation']}")
    print("-" * 40)


--- Aanbevolen Modules ---

1. molecular modeling & data-driven analysis (Score: 0.58)
   Locatie: breda
   Tags: ['moleculair', 'modelleren', 'virtual', 'screening', 'dataanalyse', 'synthese', 'small', 'molecules', 'programmeren', 'python', 'computational', 'chemistry', 'machine', 'learning']
   Waarom: ✅ **Locatie**: Deze module is in breda, wat je voorkeur heeft. 🎯 **Onderwerp**: Matcht met je interesse in: data, programmeren.
----------------------------------------
2. systems programming in c++ (Score: 0.55)
   Locatie: den bosch
   Tags: ['projectaanpak', 'programmeren', 'software', 'c', 'solid']
   Waarom: 📍 **Locatie**: Let op, deze module is in den bosch (je voorkeur was Breda). 🎯 **Onderwerp**: Matcht met je interesse in: software, programmeren.
----------------------------------------
3. datagedreven besluitvorming met ai  (Score: 0.42)
   Locatie: breda
   Tags: ['data', 'science', 'ai', 'python', 'onderzoek']
   Waarom: ✅ **Locatie**: Deze module is in breda, wat je voork